# TFOLEY: doğrudan sayısal RMS kontrolü
Mevcut TFOLEY `.venv` kernelini kullanın; AudioLDM ortamı gerekmez. Kurulum: `pip install -r requirements.txt -r requirements-tfoley.txt` (PyTorch mevcut olmalı). Model 4 saniye, 22050 Hz ve 690 RMS karesi kullanır. RMS dijital genlik ölçüsüdür; algısal loudness değildir. Peak RMS, bütün kaydın RMS'i değildir. `onsets` Gaussian enerji darbelerinin merkezleridir; gerçek ayak basma zamanı garantisi değildir. Kullanıcı metni veya referans ses embedding'i kullanılmaz.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
assert (ROOT / "src/control").exists(), "Notebooku sound-gen/notebooks altında açın."
sys.path.insert(0, str(ROOT))

In [ ]:
from src.control.tfoley import pulse_curve, CAPABILITIES
from src.control.generate import generate
CAPABILITIES

In [ ]:
import matplotlib.pyplot as plt
rms = pulse_curve(level=0.03, onsets=(0.7, 1.9, 3.1), decay_s=0.12)
plt.plot(np.arange(690) * 128 / 22050, rms)
plt.xlabel("Zaman (s)"); plt.ylabel("Hedef RMS");

`generate` kontrol eğrisini doğrudan pretrained diffusion modelinin koşul tensörüne verir. İlk koşum kaynak/548 MB checkpoint indirir. Çıktıya hedef RMS'i tutturacak gain uygulanmaz; resmî 20 Hz high-pass korunur. Centroid veya pitch girdisi bu modelde desteklenmez.

In [ ]:
path = generate("tfoley", ROOT, rms=rms, name="local_numeric_rms", seed=42, steps=50, class_name="Footstep", device="auto", prepare=True)
import soundfile as sf
y, sr = sf.read(path)
display(Audio(y, rate=sr, normalize=False))
json.loads((ROOT / "experiments/baselines/tfoley/local_numeric_rms_run.json").read_text())

Aşağıdaki tarama üç eğri ve iki seed ile altı ses üretir. RMS peak eşleştirmesi bire birdir, kabul aralığı 150 ms'dir. Peak'ler gerçek ses olayları olarak etiketlenmez. GPU/CPU ve paket farkları sonuçları değiştirebilir.

In [ ]:
from src.control.tfoley_sweep import run
report = run(ROOT, steps=50, device="auto")
report["high_over_low_rms"]